# Tree rerankers from scratch

This notebook demonstrates the three standalone GBT-style rerankers used by the project: XGBoost, LightGBM, and CatBoost. It does not import the app implementation or saved model files. The scoring functions below are small hand-written tree ensembles over a toy restaurant ranking dataset.

In [1]:
from math import log2

user = {
    "id": "u_demo",
    "favorite_categories": {"coffee", "brunch", "vietnamese"},
    "preferred_price": 2,
    "max_distance_km": 4.0,
    "history_categories": ["coffee", "brunch", "bakery", "vietnamese", "coffee"],
}

candidates = [
    {"id": "b1", "name": "Morning Bean", "categories": {"coffee", "bakery"}, "price": 2, "rating": 4.7, "distance_km": 0.7, "review_count": 410, "open_now": 1, "label": 3},
    {"id": "b2", "name": "Pho Corner", "categories": {"vietnamese", "noodles"}, "price": 1, "rating": 4.5, "distance_km": 2.2, "review_count": 260, "open_now": 1, "label": 3},
    {"id": "b3", "name": "Quiet Study Cafe", "categories": {"coffee", "dessert"}, "price": 2, "rating": 4.1, "distance_km": 1.1, "review_count": 95, "open_now": 1, "label": 2},
    {"id": "b4", "name": "Late Night BBQ", "categories": {"bbq", "korean"}, "price": 3, "rating": 4.6, "distance_km": 6.5, "review_count": 520, "open_now": 1, "label": 0},
    {"id": "b5", "name": "Campus Banh Mi", "categories": {"vietnamese", "sandwich"}, "price": 1, "rating": 4.2, "distance_km": 0.4, "review_count": 140, "open_now": 0, "label": 2},
    {"id": "b6", "name": "Premium Steak Loft", "categories": {"steak", "wine"}, "price": 4, "rating": 4.8, "distance_km": 3.6, "review_count": 310, "open_now": 1, "label": 1},
    {"id": "b7", "name": "Weekend Brunch Lab", "categories": {"brunch", "coffee"}, "price": 2, "rating": 4.4, "distance_km": 3.2, "review_count": 180, "open_now": 1, "label": 3},
    {"id": "b8", "name": "Burger Dock", "categories": {"burger", "fast food"}, "price": 2, "rating": 3.9, "distance_km": 1.8, "review_count": 220, "open_now": 1, "label": 1},
]


In [2]:
def features(user, item):
    overlap = len(user["favorite_categories"] & item["categories"])
    category_match = overlap / max(1, len(user["favorite_categories"]))
    price_match = 1.0 - min(abs(user["preferred_price"] - item["price"]), 3) / 3
    distance_fit = max(0.0, 1.0 - item["distance_km"] / user["max_distance_km"])
    rating_norm = item["rating"] / 5.0
    popularity = min(item["review_count"], 500) / 500
    history_hits = sum(1 for c in user["history_categories"] if c in item["categories"])
    history_similarity = history_hits / max(1, len(user["history_categories"]))
    return {
        "category_match": category_match,
        "price_match": price_match,
        "distance_fit": distance_fit,
        "rating_norm": rating_norm,
        "popularity": popularity,
        "open_now": item["open_now"],
        "history_similarity": history_similarity,
    }

def dcg(labels):
    return sum((2 ** label - 1) / log2(rank + 2) for rank, label in enumerate(labels))

def ndcg(ranked_items, k):
    actual = [item["label"] for item in ranked_items[:k]]
    ideal = sorted((item["label"] for item in ranked_items), reverse=True)[:k]
    return dcg(actual) / dcg(ideal) if dcg(ideal) else 0.0

def rank_with(score_fn):
    scored = []
    for item in candidates:
        f = features(user, item)
        scored.append({**item, "features": f, "score": round(score_fn(f), 4)})
    return sorted(scored, key=lambda row: row["score"], reverse=True)


In [3]:
def xgboost_like_score(f):
    score = 0.0
    score += 0.35 if f["category_match"] >= 0.34 else -0.08
    score += 0.25 if f["history_similarity"] >= 0.25 else 0.02
    score += 0.18 if f["distance_fit"] >= 0.45 else -0.05
    score += 0.12 if f["rating_norm"] >= 0.88 else -0.02
    score += 0.08 if f["price_match"] >= 0.66 else -0.04
    score += 0.04 * f["popularity"] + 0.03 * f["open_now"]
    return score

def lightgbm_like_score(f):
    score = 0.0
    score += 0.30 if f["distance_fit"] >= 0.55 else -0.02
    score += 0.22 if f["category_match"] >= 0.34 else -0.06
    score += 0.16 if f["price_match"] >= 0.66 else -0.03
    score += 0.10 if f["open_now"] else -0.05
    score += 0.06 * f["rating_norm"] + 0.06 * f["history_similarity"]
    return score

def catboost_like_score(f):
    score = 0.0
    score += 0.26 if f["price_match"] >= 0.99 and f["open_now"] else 0.02
    score += 0.18 if f["category_match"] >= 0.34 else -0.04
    score += 0.12 if f["distance_fit"] >= 0.70 else -0.02
    score += 0.08 * f["rating_norm"] + 0.05 * f["popularity"]
    return score


In [4]:
for name, fn in {
    "xgboost": xgboost_like_score,
    "lightgbm": lightgbm_like_score,
    "catboost": catboost_like_score,
}.items():
    ranked = rank_with(fn)
    print(f"\n{name.upper()}")
    print("top-5:", [(row["id"], row["name"], row["score"], row["label"]) for row in ranked[:5]])
    print("ndcg@3", round(ndcg(ranked, 3), 4), "ndcg@5", round(ndcg(ranked, 5), 4), "ndcg@10", round(ndcg(ranked, 10), 4))



XGBOOST
top-5: [('b7', 'Weekend Brunch Lab', 0.7944, 3), ('b1', 'Morning Bean', 0.6128, 3), ('b3', 'Quiet Study Cafe', 0.4476, 2), ('b8', 'Burger Dock', 0.2276, 1), ('b5', 'Campus Banh Mi', 0.1912, 2)]
ndcg@3 0.8659 ndcg@5 0.8353 ndcg@10 0.95

LIGHTGBM
top-5: [('b1', 'Morning Bean', 0.5924, 3), ('b3', 'Quiet Study Cafe', 0.5732, 2), ('b7', 'Weekend Brunch Lab', 0.5488, 3), ('b8', 'Burger Dock', 0.5468, 1), ('b5', 'Campus Banh Mi', 0.4124, 2)]
ndcg@3 0.8308 ndcg@5 0.8051 ndcg@10 0.9299

CATBOOST
top-5: [('b7', 'Weekend Brunch Lab', 0.5084, 3), ('b1', 'Morning Bean', 0.4562, 3), ('b3', 'Quiet Study Cafe', 0.4151, 2), ('b8', 'Burger Dock', 0.2844, 1), ('b5', 'Campus Banh Mi', 0.1812, 2)]
ndcg@3 0.8659 ndcg@5 0.8353 ndcg@10 0.9441
